## Extracting search events from logs

In [62]:
from driver import assign_topic_number
import glob
from tqdm.notebook import tqdm
import json
import pandas as pd

event_mapping = {
    "TaskStarted": "TS", # Task started
    "querySubmitted": "QS", # submitted user formulated query
    "searchResultGenerated": "SG", # serp generated
    "searchNoResults": "N_SG", # no SERP generated
    "clickedDidYouMeanSuggestion": "QC", # submitted SE corrected query
    "resultExposureStarted": "RV", # SERP result viewing
    "clickedResult": "RC", # SERP result clicked
    "OpenedAnswerBox": "AW", # answer writing
    "TaskEnded": "TE", # Task ended
}

event_sequences = []
relevance_df = pd.read_csv("Data/relevance.csv")

for file_name in tqdm(glob.glob("Data/3rd/*_FULL.log"), total=len(glob.glob("Data/3rd/*_FULL.log"))):
    with open(file_name, 'r') as f:
        log_entries = [json.loads(x) for x in f.readlines()]
    
    for entry in log_entries:
        if entry["type"] in list(event_mapping.keys()):
            uid = int(entry["uid"])
            topic_num = assign_topic_number(uid, entry["actual_topic_number"])
            event_type = entry["type"]
            event_key = event_mapping[event_type]
            event_time = entry["timestamp"]
            if (event_key == "RV") or (event_key == "RC"):
                url = entry["url"]
                relevant_resources = relevance_df.loc[relevance_df["topic_number"]==topic_num]["doc"].tolist()
                if url in relevant_resources:
                    event_key = "R_" + event_key
                else:
                    event_key = "NR_" + event_key
            if (event_key == "SG") and (int(entry["rank"]) > 1):
                continue
            event_sequences.append([uid, topic_num, event_type, event_key, event_time])

for file_name in tqdm(glob.glob("Data/4th/*_FULL.log"), total=len(glob.glob("Data/4th/*_FULL.log"))):
    with open(file_name, 'r') as f:
        log_entries = [json.loads(x) for x in f.readlines()]
    
    for entry in log_entries:
        if entry["type"] in list(event_mapping.keys()):
            uid = int(entry["uid"])
            topic_num = assign_topic_number(uid, entry["actual_topic_number"])
            event_type = entry["type"]
            event_key = event_mapping[event_type]
            event_time = entry["timestamp"]
            if (event_key == "RV") or (event_key == "RC"):
                url = entry["url"]
                relevant_resources = relevance_df.loc[relevance_df["topic_number"]==topic_num]["doc"].tolist()
                if url in relevant_resources:
                    event_key = "R_" + event_key
                else:
                    event_key = "NR_" + event_key
            if (event_key == "SG") and (int(entry["rank"]) > 1):
                continue
            event_sequences.append([uid, topic_num, event_type, event_key, event_time])

event_sequences_df = pd.DataFrame(event_sequences, columns=["uid", "topic_number", "action_name", "action_code", "timestamp"])   
event_sequences_df["timestamp"] = pd.to_datetime(event_sequences_df["timestamp"])   
event_sequences_df = event_sequences_df.sort_values(by=["uid", "topic_number", "timestamp"])
event_sequences_df.to_csv("Data/event_sequences_df_final.csv", index=False)

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]

In [64]:
df = event_sequences_df.sort_values(
    ["uid", "topic_number", "timestamp"]
).copy()

# Create run IDs for consecutive identical actions
df["run_id"] = (
    df.groupby(["uid", "topic_number"])["action_code"]
      .transform(lambda x: x.ne(x.shift()).cumsum())
)

# Aggregate each run
condensed_df = (
    df.groupby(["uid", "topic_number", "run_id"])
      .agg(
          action_name=("action_name", "first"),
          action_code=("action_code", "first"),
          timestamp=("timestamp", "first"),
          count=("action_code", "size")
      )
      .reset_index()
)

# Append "+" where run length > 1
condensed_df["action_code"] = condensed_df.apply(
    lambda row: f"{row['action_code']}+" if row["count"] > 1 else row["action_code"],
    axis=1
)
condensed_df["action_code"] = condensed_df["action_code"].replace("QS+", "QS")
condensed_df = condensed_df.drop(columns=["run_id"]).sort_values(by=["uid", "topic_number", "timestamp"])
condensed_df.to_csv("Data/condensed_event_sequences_df_final.csv", index=False)

In [65]:
condensed_df.loc[condensed_df["uid"]==1]

,uid,topic_number,action_name,action_code,timestamp,count
0,1,g3t1,TaskStarted,TS,2026-05-19 15:06:45.708,1
1,1,g3t1,querySubmitted,QS,2026-05-19 15:08:39.525,1
2,1,g3t1,searchResultGenerated,SG,2026-05-19 15:08:41.314,1
3,1,g3t1,resultExposureStarted,R_RV+,2026-05-19 15:08:41.585,2
4,1,g3t1,clickedResult,R_RC,2026-05-19 15:08:58.405,1
5,1,g3t1,resultExposureStarted,R_RV+,2026-05-19 15:09:35.824,2
6,1,g3t1,OpenedAnswerBox,AW,2026-05-19 15:09:38.065,1
7,1,g3t1,TaskEnded,TE,2026-05-19 15:10:10.812,1
8,1,g3t2,TaskStarted,TS,2026-05-19 14:56:45.133,1
9,1,g3t2,querySubmitted,QS,2026-05-19 14:57:07.106,2


In [66]:
event_sequences_df.loc[event_sequences_df["uid"]==1]

,uid,topic_number,action_name,action_code,timestamp
227,1,g3t1,TaskStarted,TS,2026-05-19 15:06:45.708
228,1,g3t1,querySubmitted,QS,2026-05-19 15:08:39.525
229,1,g3t1,searchResultGenerated,SG,2026-05-19 15:08:41.314
230,1,g3t1,resultExposureStarted,R_RV,2026-05-19 15:08:41.585
231,1,g3t1,resultExposureStarted,R_RV,2026-05-19 15:08:41.588
232,1,g3t1,clickedResult,R_RC,2026-05-19 15:08:58.405
233,1,g3t1,resultExposureStarted,R_RV,2026-05-19 15:09:35.824
234,1,g3t1,resultExposureStarted,R_RV,2026-05-19 15:09:35.828
235,1,g3t1,OpenedAnswerBox,AW,2026-05-19 15:09:38.065
236,1,g3t1,TaskEnded,TE,2026-05-19 15:10:10.812


## Identifying common sequence patterns

In [77]:
import pandas as pd

event_sequences_df = pd.read_csv("Data/condensed_event_sequences_df_final.csv")
final_answer_df = pd.read_csv("Data/final_answer_df.csv")

In [78]:
event_sequences_df.head()

,sequence_id,uid,topic_number,event_id,num_events,action_key,action_name,action_code,timestamp,search_success
0,1,1,g3t1,1,1,1,TaskStarted,TS,2026-05-19 15:06:45.708,1
1,1,1,g3t1,2,1,2,querySubmitted,QS,2026-05-19 15:08:39.525,1
2,1,1,g3t1,3,1,3,searchResultGenerated,SG,2026-05-19 15:08:41.314,1
3,1,1,g3t1,4,1,4,resultExposureStarted,R_RV+,2026-05-19 15:08:41.585,1
4,1,1,g3t1,5,1,5,clickedResult,R_RC,2026-05-19 15:08:58.405,1


In [79]:
event_sequences_df["uid"].unique()

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18])

In [80]:
action_key_mapping = enumerate(event_sequences_df["action_code"].unique(), start=1)
action_key_mapping = {code: key for key, code in action_key_mapping}
action_key_mapping

{'TS': 1,
 'QS': 2,
 'SG': 3,
 'R_RV+': 4,
 'R_RC': 5,
 'AW': 6,
 'TE': 7,
 'R_RV': 8,
 'AW+': 9,
 'NR_RV': 10,
 'NR_RV+': 11,
 'NR_RC': 12,
 'N_SG': 13,
 'QC': 14}

In [81]:
sequence_id_list = []
event_id_list = []
num_events_list = []
action_key_list = []
search_success_list = []

seq_id = 1
for uid in event_sequences_df["uid"].unique():
    user_df = event_sequences_df[event_sequences_df["uid"]==uid]
    for topic_num in user_df["topic_number"].unique():
        answer_match_ratio = final_answer_df[(final_answer_df["uid"]==uid) & (final_answer_df["topic_number"]==topic_num)]["answer_match_ratio"].values[0]
        # print(f"uid: {uid}, topic_num: {topic_num}, answer_match_ratio: {answer_match_ratio}")
        topic_df = user_df[user_df["topic_number"]==topic_num]
        
        evnt_id = 1
        for index, row in topic_df.iterrows():
            sequence_id_list.append(seq_id)
            event_id_list.append(evnt_id)
            num_events_list.append(1)
            action_key_list.append(action_key_mapping[row["action_code"]])
            if answer_match_ratio == 1.0:
                search_success_list.append(1)
            else:
                search_success_list.append(0)
            
            evnt_id+=1
        
        seq_id+=1

event_sequences_df["sequence_id"] = sequence_id_list
event_sequences_df["event_id"] = event_id_list 
event_sequences_df["num_events"] = num_events_list
event_sequences_df["action_key"] = action_key_list
event_sequences_df["search_success"] = search_success_list

event_sequences_df = event_sequences_df[["sequence_id", "uid", "topic_number", "event_id", "num_events", "action_key", "action_name", "action_code", "timestamp", "search_success"]]
event_sequences_df.to_csv("Data/condensed_event_sequences_df_final.csv", index=False)

In [82]:
event_sequences_df.loc[event_sequences_df["action_name"]=="TaskStarted"]

,sequence_id,uid,topic_number,event_id,num_events,action_key,action_name,action_code,timestamp,search_success
0,1,1,g3t1,1,1,1,TaskStarted,TS,2026-05-19 15:06:45.708,1
8,2,1,g3t2,1,1,1,TaskStarted,TS,2026-05-19 14:56:45.133,0
15,3,1,g3t3,1,1,1,TaskStarted,TS,2026-05-19 14:48:04.522,0
27,4,2,g3t1,1,1,1,TaskStarted,TS,2026-05-19 15:11:02.982,1
34,5,2,g3t2,1,1,1,TaskStarted,TS,2026-05-19 14:58:35.224,0
46,6,2,g3t3,1,1,1,TaskStarted,TS,2026-05-19 14:48:38.961,1
60,7,3,g3t1,1,1,1,TaskStarted,TS,2026-05-19 14:48:36.775,1
66,8,3,g3t2,1,1,1,TaskStarted,TS,2026-05-19 15:00:47.002,0
80,9,3,g3t3,1,1,1,TaskStarted,TS,2026-05-19 14:51:43.431,1
89,10,4,g3t1,1,1,1,TaskStarted,TS,2026-05-19 15:19:44.569,1


In [ ]:
from pycspade.helpers import spade, print_result

import pandas as pd

def is_sublist(sub, main):
    n = len(sub)
    if n == 0:
        return True  # An empty list is always a subset
    return any(main[i : i + n] == sub for i in range(len(main) - n + 1))

def save_extracted_sequences(result, action_key_mapping, prune=False):
    reverse_action_key_mapping = {v: k for k, v in action_key_mapping.items()}
    rows = []

    for line in result["seqstrm"].splitlines():
        if not line.strip():
            continue

        pattern, counts = line.split(" -- ")

        occurs, accum = map(int, counts.split())

        decoded = " -> ".join(
            reverse_action_key_mapping[int(x.strip())]
            for x in pattern.split("->")
        )

        rows.append({
            "Sequence": pattern,
            "Decoded_sequence": decoded,
            "Actions": [int(x.strip()) for x in pattern.split("->")],
            "Num_actions": len(decoded.split("->")),
            "Occurs": occurs, # number of sessions the pattern occurs in
            "Accum": accum, # total number of times the pattern occurs across all sessions
            "Support": occurs / result["nsequences"]
        })

    df = pd.DataFrame(rows)
    # df = df.loc[df["Num_actions"] > 1].reset_index(drop=True)
    
    if prune == True:
        closed = []
        # maximal = []

        for _, row in df.iterrows():
            actions = row["Actions"]
            num_actions = row["Num_actions"]
            support = row["Support"]

            is_closed_flag = True
            candidate_patterns = df.loc[(df["Support"] == support) & (df["Num_actions"] > num_actions)] # all patterns longer than current pattern with same support

            if len(candidate_patterns) > 0:
                for _, candidate in candidate_patterns.iterrows():
                    # print(len(actions), len(candidate["Actions"]))
                    if is_sublist(actions, candidate["Actions"]):
                        is_closed_flag = False
                        # is_maximal_flag = False

            closed.append(is_closed_flag)

        df["is_closed"] = closed
        df = df.loc[df["is_closed"]==True].drop(columns=["is_closed"]).reset_index(drop=True)

    df = df.sort_values(by=["Support", "Num_actions"], ascending=False).reset_index(drop=True)

    return df

In [101]:
event_sequences_df.to_csv(f"Data/all_sessions.txt", sep=" ", header=False, index=False, columns=["sequence_id", "event_id", "num_events", "action_key"])
all_sessions_events = spade(filename=f"Data/all_sessions.txt", support=0.2, parse=True, maxgap=1)
all_sessions_freq_seq_pruned = save_extracted_sequences(all_sessions_events, action_key_mapping, prune=False)
all_sessions_freq_seq_pruned.to_csv("Results/all_sessions_freq_seq.csv", index=False)
all_sessions_freq_seq_pruned

,Sequence,Decoded_sequence,Actions,Num_actions,Occurs,Accum,Support
0,2 -> 3,QS -> SG,"[2, 3]",2,54,54,1.000000
1,1 -> 2,TS -> QS,"[1, 2]",2,54,54,1.000000
2,1,TS,[1],1,54,54,1.000000
3,2,QS,[2],1,54,54,1.000000
4,3,SG,[3],1,54,54,1.000000
5,7,TE,[7],1,54,54,1.000000
6,1 -> 2 -> 3,TS -> QS -> SG,"[1, 2, 3]",3,50,50,0.925926
7,4,R_RV+,[4],1,50,50,0.925926
8,2 -> 3 -> 4,QS -> SG -> R_RV+,"[2, 3, 4]",3,47,47,0.870370
9,3 -> 4,SG -> R_RV+,"[3, 4]",2,47,47,0.870370


In [ ]:
for task in ["t1", "t2", "t3"]:
    g3_topic_num = "g3"+task
    g4_topic_num = "g4"+task
    
    task_sessions = event_sequences_df.loc[(event_sequences_df["topic_number"]==g3_topic_num) | (event_sequences_df["topic_number"]==g4_topic_num)]

    # print(task_sessions["topic_number"].unique())

    successful_events = task_sessions[task_sessions["search_success"]==1]
    successful_events.to_csv(f"Data/successful_sessions_task_{task}.txt", sep=" ", header=False, index=False, columns=["sequence_id", "event_id", "num_events", "action_key"])

    failed_events = task_sessions[task_sessions["search_success"]==0]
    failed_events.to_csv(f"Data/failed_sessions_task_{task}.txt", sep=" ", header=False, index=False, columns=["sequence_id", "event_id", "num_events", "action_key"])

    # To get raw SPADE output
    successful_result = spade(filename=f"Data/successful_sessions_task_{task}.txt", support=0.2, parse=True, maxgap=1)
    failed_result = spade(filename=f"Data/failed_sessions_task_{task}.txt", support=0.2, parse=True, maxgap=1)

    successful_df_pruned = save_extracted_sequences(successful_result, action_key_mapping, prune=True)
    successful_df_all = save_extracted_sequences(successful_result, action_key_mapping, prune=False)
    failed_df_pruned = save_extracted_sequences(failed_result, action_key_mapping, prune=True)
    failed_df_all = save_extracted_sequences(failed_result, action_key_mapping, prune=False)

    successful_df_pruned.to_csv(f"Results/successful_patterns_pruned_task_{task}.csv", index=False)
    successful_df_all.to_csv(f"Results/successful_patterns_all_task_{task}.csv", index=False)

    failed_df_pruned.to_csv(f"Results/failed_patterns_pruned_task_{task}.csv", index=False)
    failed_df_all.to_csv(f"Results/failed_patterns_all_task_{task}.csv", index=False)

    events_pruned = []

    for _, row in successful_df_pruned.iterrows():
        sequence = row["Decoded_sequence"]
        success_support = row["Support"]
        success_accum = row["Accum"]
        failed_row = failed_df_all[failed_df_all["Decoded_sequence"]==sequence]
        if len(failed_row) == 0:
            failed_support = 0
            failed_accum = 0
        elif len(failed_row) == 1:
            failed_support = failed_row["Support"].values[0]
            failed_accum = failed_row["Accum"].values[0]
        else:
            print(sequence, failed_row)
        events_pruned.append({
            "sequence": sequence,
            "successful_support": success_support,
            # "successul_accum": success_accum,
            "failed_support": failed_support,
            # "failed_accum": failed_accum
        })

    for _, row in failed_df_pruned.iterrows():
        sequence = row["Decoded_sequence"]
        failed_support = row["Support"]
        failed_accum = row["Accum"]
        success_row = successful_df_all[successful_df_all["Decoded_sequence"]==sequence]
        
        if len(success_row) == 0:
            success_support = 0
            success_accum = 0
        elif len(success_row) == 1:
            success_support = success_row["Support"].values[0]
            success_accum = success_row["Accum"].values[0]
        else:
            print(sequence, success_row)
        events_pruned.append({
            "sequence": sequence,
            "successful_support": success_support,
            # "successul_accum": success_accum,
            "failed_support": failed_support,
            # "failed_accum": failed_accum
        })

    events_pruned_df = pd.DataFrame(events_pruned)
    events_pruned_df.drop_duplicates(subset=["sequence"], inplace=True)
    events_pruned_df = events_pruned_df.sort_values(by=["failed_support"], ascending=False).reset_index(drop=True)
    events_pruned_df.to_csv(f"Results/pruned_patterns_combined_task_{task}.csv", index=False)

In [90]:
import numpy as np

final_answer_df["TaskStartTime"] = pd.to_datetime(final_answer_df["TaskStartTime"])
final_answer_df["TaskEndTime"] = pd.to_datetime(final_answer_df["TaskEndTime"])

for task in ["t1", "t2", "t3"]:
    g3_topic_num = "g3"+task
    g4_topic_num = "g4"+task
    
    task_sessions = final_answer_df.loc[(final_answer_df["topic_number"]==g3_topic_num) | (final_answer_df["topic_number"]==g4_topic_num)]
    success_task = task_sessions.loc[task_sessions["answer_match_ratio"]==1.0]
    failed_task = task_sessions.loc[task_sessions["answer_match_ratio"]<1.0]

    success_task_times = [(row["TaskEndTime"] - row["TaskStartTime"]).total_seconds() for _, row in success_task.iterrows()]
    failed_task_times = [(row["TaskEndTime"] - row["TaskStartTime"]).total_seconds() for _, row in failed_task.iterrows()]

    avg_success_time = np.mean(success_task_times)/60
    avg_failed_time = np.mean(np.mean(failed_task_times))/60

    print(f"Task: {task}; avg success time: {avg_success_time} minutes; avg failed time: {avg_failed_time} minutes")



Task: t1; avg success time: 5.278418888888888 minutes; avg failed time: 6.4089222222222215 minutes
Task: t2; avg success time: 11.940798148148145 minutes; avg failed time: 8.838687037037037 minutes
Task: t3; avg success time: 12.334844444444444 minutes; avg failed time: 12.636077777777777 minutes


In [91]:
final_answer_df.loc[(final_answer_df["topic_number"] == "g3t1") | (final_answer_df["topic_number"]=="g4t1")]

,uid,topic_number,prompt,answer,TaskStartTime,TaskEndTime,TaskDuration,num_answer_matches,matched_keywords,answer_match_ratio
2,1,g3t1,Come si chiama il fiume sacro agli indiani?,Il fiume gange,2026-05-19 15:06:45.708,2026-05-19 15:10:10.812,3.418400,1,['gange'],1.00
5,2,g3t1,Come si chiama il fiume sacro agli indiani?,Fiume gange,2026-05-19 15:11:02.982,2026-05-19 15:15:05.518,4.042267,1,['gange'],1.00
6,3,g3t1,Come si chiama il fiume sacro agli indiani?,Gange,2026-05-19 14:48:36.775,2026-05-19 14:51:32.588,2.930217,1,['gange'],1.00
11,4,g3t1,Come si chiama il fiume sacro agli indiani?,Gange,2026-05-19 15:19:44.569,2026-05-19 15:21:00.731,1.269367,1,['gange'],1.00
12,5,g3t1,Come si chiama il fiume sacro agli indiani?,"Gange, il fiume sacro agli indu e la popolazio...",2026-05-19 14:48:44.193,2026-05-19 14:58:33.130,9.815617,1,['gange'],1.00
15,6,g3t1,Come si chiama il fiume sacro agli indiani?,Gange,2026-05-19 14:48:59.703,2026-05-19 14:52:28.940,3.487283,1,['gange'],1.00
20,7,g3t1,Come si chiama il fiume sacro agli indiani?,Gange il fiume sacro again indu e alla popolaz...,2026-05-19 15:00:08.029,2026-05-19 15:05:34.776,5.445783,1,['gange'],1.00
23,8,g3t1,Come si chiama il fiume sacro agli indiani?,Gange,2026-05-19 15:15:25.308,2026-05-19 15:16:38.128,1.213667,1,['gange'],1.00
25,9,g4t1,Come si chiamano i satelliti maggiori di Giove?,Ganimede e Callisto,2026-05-19 10:51:13.569,2026-05-19 11:00:07.936,8.906117,2,"['ganimede', 'callisto']",0.50
29,10,g4t1,Come si chiamano i satelliti maggiori di Giove?,Io Europa ganimede e calisto,2026-05-19 11:00:44.866,2026-05-19 11:05:41.810,4.949067,3,"['io', 'europa', 'ganimede']",0.75


In [92]:
final_answer_df.loc[(final_answer_df["topic_number"] == "g3t2") | (final_answer_df["topic_number"]=="g4t2")]

,uid,topic_number,prompt,answer,TaskStartTime,TaskEndTime,TaskDuration,num_answer_matches,matched_keywords,answer_match_ratio
1,1,g3t2,Qual è il nome del simbolo al centro della ban...,Il simbolo della bandiera rappresenta una ruota,2026-05-19 14:56:45.133,2026-05-19 15:06:18.848,9.561917,1,['ruota'],0.5
4,2,g3t2,Qual è il nome del simbolo al centro della ban...,Ruota,2026-05-19 14:58:35.224,2026-05-19 15:10:30.790,11.926100,1,['ruota'],0.5
8,3,g3t2,Qual è il nome del simbolo al centro della ban...,Un sole,2026-05-19 15:00:47.002,2026-05-19 15:08:53.840,8.113967,0,[],0.0
9,4,g3t2,Qual è il nome del simbolo al centro della ban...,Ashoka. La ruota della vita buddista,2026-05-19 14:48:41.489,2026-05-19 15:13:24.507,24.716967,2,"['ashoka', 'ruota']",1.0
14,5,g3t2,Qual è il nome del simbolo al centro della ban...,Bandiera indiana significato ashoka chakra sim...,2026-05-19 15:13:26.617,2026-05-19 15:22:57.085,9.507800,1,['ashoka'],0.5
17,6,g3t2,Qual è il nome del simbolo al centro della ban...,La ruota ashoka chakra,2026-05-19 15:01:28.680,2026-05-19 15:16:06.067,14.623117,2,"['ashoka', 'ruota']",1.0
18,7,g3t2,Qual è il nome del simbolo al centro della ban...,E un. Sole blu,2026-05-19 14:48:10.578,2026-05-19 14:53:59.548,5.816167,0,[],0.0
22,8,g3t2,Qual è il nome del simbolo al centro della ban...,"Ruota con 24 raggi,blu nevy",2026-05-19 15:03:02.463,2026-05-19 15:14:59.118,11.944250,1,['ruota'],0.5
24,9,g4t2,Qual è stato il primo animale nello spazio e c...,Il primo animale mandato nello spazio è state...,2026-05-19 10:41:34.057,2026-05-19 10:51:05.571,9.525233,1,['Laika'],0.5
28,10,g4t2,Qual è stato il primo animale nello spazio e c...,Il primo animale che è andato nello spazio er...,2026-05-19 10:52:44.731,2026-05-19 11:00:30.093,7.756033,2,"['cane', 'Laika']",1.0


In [93]:
final_answer_df.loc[(final_answer_df["topic_number"] == "g3t3") | (final_answer_df["topic_number"]=="g4t3")]

,uid,topic_number,prompt,answer,TaskStartTime,TaskEndTime,TaskDuration,num_answer_matches,matched_keywords,answer_match_ratio
0,1,g3t3,Quali sono 5 animali tipici dell'India?,Rinoceronte tigre Leone stambecco cervo,2026-05-19 14:48:04.522,2026-05-19 14:56:19.153,8.243850,4,"['tigre', 'rinoceronte', 'leone', 'cervo']",0.80
3,2,g3t3,Quali sono 5 animali tipici dell'India?,"Tigre,leone,rinoceronte,barasingha,blackbuck.",2026-05-19 14:48:38.961,2026-05-19 14:58:07.578,9.476950,5,"['tigre', 'rinoceronte', 'leone', 'barasingha'...",1.00
7,3,g3t3,Quali sono 5 animali tipici dell'India?,"Tigre ,Leone, rinoceronte ,Blackbuck barasingha",2026-05-19 14:51:43.383,2026-05-19 15:00:37.615,8.903867,5,"['tigre', 'rinoceronte', 'leone', 'barasingha'...",1.00
10,4,g3t3,Quali sono 5 animali tipici dell'India?,Tigre del bengaline\nLeone asiatico\nRinoceron...,2026-05-19 15:13:35.566,2026-05-19 15:19:38.451,6.048083,5,"['tigre', 'rinoceronte', 'leone', 'barasingha'...",1.00
13,5,g3t3,Quali sono 5 animali tipici dell'India?,Natura selvaggia tra tigri e leoni terrelncogn...,2026-05-19 14:59:37.249,2026-05-19 15:13:16.143,13.648233,0,[],0.00
16,6,g3t3,Quali sono 5 animali tipici dell'India?,Tigress del bengala Leone asiatico rinoceronte...,2026-05-19 14:52:48.947,2026-05-19 15:00:34.997,7.767500,5,"['tigre', 'rinoceronte', 'leone', 'barasingha'...",1.00
19,7,g3t3,Quali sono 5 animali tipici dell'India?,Bengala Leone asiatico rinoceronte indiano ba...,2026-05-19 14:54:18.067,2026-05-19 14:59:45.569,5.458367,2,"['rinoceronte', 'leone']",0.40
21,8,g3t3,Quali sono 5 animali tipici dell'India?,Tigre elefante pavone rinoceronte scimmie,2026-05-19 14:49:20.685,2026-05-19 15:01:54.523,12.563967,4,"['tigre', 'rinoceronte', 'elefante', 'pavone']",0.80
26,9,g4t3,Quali sono i nomi di altre 3 galassie oltre la...,Andromeda Triangolo Orsa Minore,2026-05-19 11:00:21.290,2026-05-19 11:20:02.362,19.684533,3,"['Andromeda', 'Triangolo', 'Orsa Minore']",1.00
27,10,g4t3,Quali sono i nomi di altre 3 galassie oltre la...,Galassia del triangolo galassia del pegaso gal...,2026-05-19 10:41:40.618,2026-05-19 10:52:22.846,10.703800,3,"['Triangolo', 'Pegaso', 'Fenice']",1.00


In [94]:
final_answer_df.loc[final_answer_df["answer_match_ratio"]==1.0]["topic_number"].value_counts()

topic_number
g3t1    10
g3t3     6
g4t3     6
g4t2     6
g4t1     5
g3t2     3
Name: count, dtype: int64

## Miscelaneous

In [ ]:
# successful_sessions = final_answer_df.loc[final_answer_df["answer_match_ratio"]==1.0]
# failed_sessions = final_answer_df.loc[final_answer_df["answer_match_ratio"]<1.0]

successful_transactions = []
failed_transactions = []

for _, row in final_answer_df.iterrows():
    uid = row["uid"]
    topic_num = row["topic_number"]
    session_actions = event_sequences_df.loc[(event_sequences_df["uid"]==uid) & (event_sequences_df["topic_number"]==topic_num)]
    if row["answer_match_ratio"] == 1.0:
        successful_transactions.append(session_actions["event_key"].tolist())
    else:
        failed_transactions.append(session_actions["event_key"].tolist())

In [ ]:
successful_transactions_lengths = [int(len(event_seq)) for event_seq in successful_transactions]
sum(successful_transactions_lengths)/len(successful_transactions_lengths)

41.30555555555556

In [ ]:
failed_transactions_lengths = [int(len(event_seq)) for event_seq in failed_transactions]
sum(failed_transactions_lengths)/len(failed_transactions_lengths)

42.388888888888886

In [ ]:
from gsppy.gsp import GSP
from prefixspan import PrefixSpan


# transactions = [
#     ["Bread", "Milk"],
#     ["Bread", "Diaper", "Beer", "Eggs"],
#     ["Milk", "Diaper", "Beer", "Coke"],
# ]

# patterns = GSP(successful_transactions).search(min_support=0.3, verbose=True)
# for level, freq_patterns in enumerate(patterns, start=1):
#     print(f"Level {level}: {freq_patterns}")

ps = PrefixSpan(successful_transactions)
patterns = ps.frequent(0.3*len(successful_transactions), closed=True) # all closed patterns occuring even once
patterns = sorted(patterns, reverse=True)
for sup, patt in patterns:
    print(f"{sup}: {' -> '.join(patt)}")

In [ ]:
ps = PrefixSpan(failed_transactions)
patterns = ps.frequent(0.3*len(failed_transactions), closed=True) # all closed patterns occuring even once
patterns = sorted(patterns, reverse=True)
for sup, patt in patterns:
    print(f"{sup}: {' -> '.join(patt)}")

18: TS -> TH -> QS -> SG -> R_RV -> AW -> TE
18: TS -> TH -> QF -> R_RV -> AW -> TE
18: TS -> TH -> QF -> QS -> SG -> R_RV -> AW -> TE
18: TS -> TH -> QF -> AW -> TE
17: TS -> TH -> TH -> QS -> SG -> R_RV -> AW -> TE
17: TS -> TH -> TH -> QF -> R_RV -> AW -> TE
17: TS -> TH -> TH -> QF -> QS -> SG -> R_RV -> AW -> TE
17: TS -> TH -> TH -> QF -> AW -> TE
17: TS -> TH -> QS -> SG -> R_RV -> R_RV -> AW -> TE
17: TS -> TH -> QF -> QS -> SG -> R_RV -> R_RV -> AW -> TE
17: TS -> TH -> QF -> QF -> R_RV -> AW -> TE
17: TS -> TH -> QF -> QF -> QS -> SG -> R_RV -> AW -> TE
17: TS -> TH -> QF -> QF -> AW -> TE
16: TS -> TH -> TH -> QS -> SG -> R_RV -> R_RV -> R_RV -> R_RV -> AW -> TE
16: TS -> TH -> TH -> QS -> SG -> R_RV -> R_RV -> R_RV -> AW -> TE
16: TS -> TH -> TH -> QS -> SG -> R_RV -> R_RV -> AW -> TE
16: TS -> TH -> TH -> QF -> QS -> SG -> R_RV -> R_RV -> R_RV -> R_RV -> AW -> TE
16: TS -> TH -> TH -> QF -> QS -> SG -> R_RV -> R_RV -> R_RV -> AW -> TE
16: TS -> TH -> TH -> QF -> QS -> SG -